# 模型推理 - 使用 QLoRA 微调后的 ChatGLM-6B

In [1]:
import torch
from transformers import AutoModel, AutoTokenizer, BitsAndBytesConfig

# 模型ID或本地路径
model_name_or_path = 'THUDM/chatglm3-6b'

In [2]:
_compute_dtype_map = {
    'fp32': torch.float32,
    'fp16': torch.float16,
    'bf16': torch.bfloat16
}

# QLoRA 量化配置
q_config = BitsAndBytesConfig(load_in_4bit=True,
                              bnb_4bit_quant_type='nf4',
                              bnb_4bit_use_double_quant=True,
                              bnb_4bit_compute_dtype=_compute_dtype_map['bf16'])
# 加载量化后模型
base_model = AutoModel.from_pretrained(model_name_or_path,
                                  quantization_config=q_config,
                                  device_map='auto',
                                  trust_remote_code=True)

Loading checkpoint shards:   0%|          | 0/7 [00:00<?, ?it/s]

In [3]:
base_model.requires_grad_(False)
base_model.eval()

ChatGLMForConditionalGeneration(
  (transformer): ChatGLMModel(
    (embedding): Embedding(
      (word_embeddings): Embedding(65024, 4096)
    )
    (rotary_pos_emb): RotaryEmbedding()
    (encoder): GLMTransformer(
      (layers): ModuleList(
        (0-27): 28 x GLMBlock(
          (input_layernorm): RMSNorm()
          (self_attention): SelfAttention(
            (query_key_value): Linear4bit(in_features=4096, out_features=4608, bias=True)
            (core_attention): CoreAttention(
              (attention_dropout): Dropout(p=0.0, inplace=False)
            )
            (dense): Linear4bit(in_features=4096, out_features=4096, bias=False)
          )
          (post_attention_layernorm): RMSNorm()
          (mlp): MLP(
            (dense_h_to_4h): Linear4bit(in_features=4096, out_features=27392, bias=False)
            (dense_4h_to_h): Linear4bit(in_features=13696, out_features=4096, bias=False)
          )
        )
      )
      (final_layernorm): RMSNorm()
    )
    (output_la

In [4]:
tokenizer = AutoTokenizer.from_pretrained(model_name_or_path, trust_remote_code=True)

Setting eos_token is not supported, use the default one.
Setting pad_token is not supported, use the default one.
Setting unk_token is not supported, use the default one.


## 使用微调前 ChatGLM3

In [5]:
input_text = "你是谁？"

In [6]:
response, history = base_model.chat(tokenizer, query=input_text)

In [7]:
print(response)

我是一个名为 ChatGLM3-6B 的人工智能助手，是基于清华大学 KEG 实验室和智谱 AI 公司于 2023 年共同训练的语言模型开发的。我的任务是针对用户的问题和要求提供适当的答复和支持。


In [8]:
response, history = base_model.chat(tokenizer, query="你叫什么？", history=history)
print(response)

我叫 ChatGLM3-6B，是清华大学 KEG 实验室和智谱 AI 公司于 2023 年共同训练的语言模型开发的人工智能助手。


## 微调前后效果对比

#### 加载 QLoRA Adapter(Epoch=3, automade-dataset(fixed))

In [9]:
from peft import PeftModel, PeftConfig

epochs = 50
timestamp = "20240312_113419"
peft_model_path = f"models/{model_name_or_path}-epoch{epochs}-{timestamp}"

config = PeftConfig.from_pretrained(peft_model_path)
model = PeftModel.from_pretrained(base_model, peft_model_path)

In [10]:
def compare_chatglm_results(query):
    base_response, base_history = base_model.chat(tokenizer, query)

    inputs = tokenizer(query, return_tensors="pt").to(0)
    ft_out = model.generate(**inputs, max_new_tokens=512)
    ft_response = tokenizer.decode(ft_out[0], skip_special_tokens=True)
    
    print(f"问题：{query}\n\n原始输出：\n{base_response}\n\n\nChatGLM3-6B(Epoch=3, automade-dataset(fixed))微调后：\n{ft_response}")
    return base_response, ft_response

In [11]:
base_response, ft_response = compare_chatglm_results(query="你好？")

问题：你好？

原始输出：
您好，我是 Solidus，一个由 Jiangze 开发的 AI 助手，很高兴认识您。请问我能为您做些什么？


ChatGLM3-6B(Epoch=3, automade-dataset(fixed))微调后：
[gMASK] sop 你好？ 您好，我是 Solidus，一个由 Jiangze 开发的 AI 助手，很高兴认识您。请问我能为您做些什么？


In [12]:
base_response, ft_response = compare_chatglm_results(query="你是谁？")

问题：你是谁？

原始输出：
您好，我是 Solidus，由 Jiangze 开发，旨在为用户提供智能化的回答和帮助。


ChatGLM3-6B(Epoch=3, automade-dataset(fixed))微调后：
[gMASK] sop 你是谁？ 您好，我是 Solidus，由 Jiangze 开发，旨在为用户提供智能化的回答和帮助。


In [13]:
base_response, ft_response = compare_chatglm_results(query="你是谁？")

问题：你是谁？

原始输出：
您好，我是 Solidus，由 Jiangze 开发，旨在为用户提供智能化的回答和帮助。


ChatGLM3-6B(Epoch=3, automade-dataset(fixed))微调后：
[gMASK] sop 你是谁？ 您好，我是 Solidus，由 Jiangze 开发，旨在为用户提供智能化的回答和帮助。
